# 03 - Agent Demo

This notebook demonstrates the LangGraph ReAct agent that uses LlamaIndex RAG
to answer questions about indexed contracts.

## Architecture
```
User Query -> LangGraph Agent -> search_contracts tool -> LlamaIndex Query Engine -> Pinecone
                    ^                                           |
                    |___________ synthesized answer ____________|
```

**Prerequisites:** Run `01_generate_data.ipynb` and `02_ingestion.ipynb` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

settings = get_settings()
init_observability(settings)

## Build the Agent

The agent consists of:
- **Azure OpenAI LLM** with tool-calling capability
- **`search_contracts`** tool wrapping the LlamaIndex query engine
- **LangGraph** state machine with ReAct routing (agent -> tools -> agent -> END)

In [ ]:
from contract_lens.agent.graph import build_agent

agent, callback_handler = build_agent(settings)
config = {'callbacks': [callback_handler]} if callback_handler else {}

print('Agent ready.')
print(f'LangFuse tracing: {"enabled" if callback_handler else "disabled"}')

## Helper Function

A utility to invoke the agent and display the full message trace.

In [ ]:
from langchain_core.messages import HumanMessage


def ask(question: str) -> str:
    """Send a question to the agent and print the conversation trace."""
    print(f'User: {question}')
    print('-' * 60)

    result = agent.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=config,
    )

    for msg in result['messages']:
        role = msg.__class__.__name__.replace('Message', '')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  [{role}] Tool call: {tc["name"]}({tc["args"]})')
        elif role == 'Tool':
            preview = msg.content[:200] + '...' if len(msg.content) > 200 else msg.content
            print(f'  [{role}] {preview}')
        elif role != 'Human':
            print(f'  [{role}] {msg.content}')

    print('-' * 60)
    answer = result['messages'][-1].content
    return answer

## Example Queries

### Query 1: General overview

In [ ]:
ask('What types of agreements are stored in the system?');

### Query 2: Specific clause lookup

In [ ]:
ask('What are the confidentiality obligations in the NDA?');

### Query 3: Pricing and financial terms

In [ ]:
ask('What are the pricing terms in the IT service agreement?');

### Query 4: Polish language agreement

In [ ]:
ask('Jakie są warunki umowy najmu biura?');

### Query 5: Cross-document comparison

In [ ]:
ask('Compare the termination clauses across all agreements.');

## Summary

The agent demonstrates:
- **Tool-calling**: The LLM autonomously decides when to search the contract knowledge base
- **Metadata filtering**: Queries can target specific languages or document types
- **Multilingual support**: Works with both English and Polish agreements
- **RAG synthesis**: LlamaIndex retrieves relevant chunks and synthesizes coherent answers
- **Observability**: All traces are captured in LangFuse (if configured)

For interactive use, run `python scripts/chat.py` from the terminal.